In [1]:
# ==============================================================================
# CYBERSENTINELS MASTER KAGGLE PIPELINE
# Amy (Data & Features) | Gowri (Model) | Aein (Cost Optimization & Submissions)
# ==============================================================================

import os, glob, time, joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, average_precision_score

print("=" * 60)
print("🚀 STARTING THE PIPELINE...")
print("=" * 60)

# --- 1. FIND THE DATASET FILES ON KAGGLE ---
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if matches: return matches[0]
    for local in [f"data/{filename}", filename, f"../data/{filename}"]:
        if os.path.exists(local): return local
    raise FileNotFoundError(f"Cannot find {filename} in /kaggle/input!")

TRAIN_PATH = find_file("train.csv")
VAL_PATH   = find_file("validation.csv")
OUT_DIR    = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
print(f"✅ Found Train: {TRAIN_PATH}")
print(f"✅ Found Validation: {VAL_PATH}")

# --- 2. AMY'S PART: LOAD & ENGINEER FEATURES ---
RAW_COLS = [
    "proto", "state", "dur", "sbytes", "dbytes", "sloss", "dloss", "service",
    "Sload", "Dload", "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb",
    "smeansz", "dmeansz", "trans_depth", "res_bdy_len", "Sjit", "Djit",
    "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat", "is_sm_ips_ports",
    "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst",
    "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm"
]
CAT_COLS = ["proto", "state", "service"]
NUM_COLS = [c for c in RAW_COLS if c not in CAT_COLS]

def load_data(path, is_labelled=True, nrows=None):
    cols = RAW_COLS + ["attack_cat", "Label"] if is_labelled else RAW_COLS
    df = pd.read_csv(path, header=None, names=cols, dtype=str, keep_default_na=False, nrows=nrows)
    if is_labelled:
        df["attack_cat"] = df["attack_cat"].str.strip().replace({"": "Normal"})
        df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(0).astype(int)
    return df

def engineer_features(df, cat_mappings=None, is_train=False):
    X = pd.DataFrame(index=df.index)
    if is_train or cat_mappings is None:
        cat_mappings = {c: {val: i for i, val in enumerate(sorted(df[c].astype(str).str.strip().unique()))} for c in CAT_COLS}
    
    # Categories to numbers (safe against unseen zero-day protocols)
    for c in CAT_COLS:
        mapping = cat_mappings[c]
        X[c] = df[c].astype(str).str.strip().map(mapping).fillna(len(mapping)).astype(np.int32)
        
    for c in NUM_COLS:
        X[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype(np.float32)
        
    # Domain Ratios (Detects attacks by flow behavior)
    X["bytes_per_src_pkt"] = (X["sbytes"] / (X["Spkts"] + 1.0)).astype(np.float32)
    X["bytes_per_dst_pkt"] = (X["dbytes"] / (X["Dpkts"] + 1.0)).astype(np.float32)
    X["byte_ratio"]        = ((X["sbytes"] + 1.0) / (X["dbytes"] + 1.0)).astype(np.float32)
    X["pkt_ratio"]         = ((X["Spkts"] + 1.0) / (X["Dpkts"] + 1.0)).astype(np.float32)
    X["loss_rate"]         = ((X["sloss"] + X["dloss"]) / (X["Spkts"] + X["Dpkts"] + 1.0)).astype(np.float32)
    X["synack_ratio"]      = (X["synack"] / (X["tcprtt"] + 1e-6)).astype(np.float32)
    X["ackdat_ratio"]      = (X["ackdat"] / (X["tcprtt"] + 1e-6)).astype(np.float32)
    X["srv_density_src"]   = (X["ct_srv_src"] / (X["ct_src_ltm"] + 1.0)).astype(np.float32)
    for c in ["sbytes", "dbytes", "Sload", "Dload", "dur"]:
        X[f"log_{c}"] = np.log1p(np.maximum(0.0, X[c])).astype(np.float32)
    return X, cat_mappings

print("\n[Amy] Loading data & engineering features...")
t0 = time.time()
train_df = load_data(TRAIN_PATH, is_labelled=True, nrows=None) 
val_df   = load_data(VAL_PATH, is_labelled=True)

X_train, cat_mappings = engineer_features(train_df, is_train=True)
X_val, _              = engineer_features(val_df, cat_mappings=cat_mappings, is_train=False)
y_train, y_val        = train_df["Label"].values, val_df["Label"].values
print(f"✅ Features ready in {time.time()-t0:.1f}s! ({X_train.shape[1]} columns)")

# --- 3. GOWRI'S PART: TRAIN LIGHTGBM ---
print("\n[Gowri] Training LightGBM model...")
t_train = time.time()
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.06,
    num_leaves=63,
    scale_pos_weight=1.5, # Gives extra weight to rare attacks
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model.fit(X_train, y_train)
print(f"✅ Model trained in {time.time()-t_train:.1f}s!")

# --- 4. AEIN'S PART: ASYMMETRIC LOSS SCAN & SAVE ---
print("\n[Aein] Finding best threshold to minimize risk penalty...")
val_probs = model.predict_proba(X_val)[:, 1]
COST_WEIGHT = 20.0 # Missed attack = 20 penalty points

# Fast scan across candidate thresholds
y_true_bool = (y_val == 1)
total_pos = int(np.count_nonzero(y_true_bool))
best_th = 0.50
min_cost = float("inf")

for th in np.linspace(0.005, 0.95, 200):
    pred_pos = (val_probs >= th)
    tp = int(np.count_nonzero(pred_pos & y_true_bool))
    fp = int(np.count_nonzero(pred_pos & (~y_true_bool)))
    cost = COST_WEIGHT * (total_pos - tp) + fp
    if cost < min_cost:
        min_cost = cost
        best_th = float(th)

# Metrics at best threshold
val_preds = (val_probs >= best_th).astype(int)
f1 = f1_score(y_val, val_preds)
pr_auc = average_precision_score(y_val, val_probs)
cm = confusion_matrix(y_val, val_preds)
tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 50)
print(f"🏆 VALIDATION RESULTS AT BEST THRESHOLD (T* = {best_th:.4f})")
print("=" * 50)
print(f"  F1-Score:    {f1:.4f}  (Target: > 0.95)")
print(f"  PR-AUC:      {pr_auc:.4f}  (Target: > 0.99)")
print(f"  Attacks Caught (TP): {tp:,} out of {tp+fn:,}")
print(f"  Missed Attacks (FN): ONLY {fn:,}!")
print(f"  Total Cost Score:    {int(min_cost):,} (Cost = 20*FN + FP)")

# Save Model Package & Validation Predictions
package = {"model": model, "cat_mappings": cat_mappings, "feature_cols": list(X_train.columns), "best_threshold": best_th}
joblib.dump(package, os.path.join(OUT_DIR, "model.joblib"))

sub_df = pd.DataFrame({"row_id": np.arange(1, len(val_preds) + 1), "prediction": val_preds, "probability": np.round(val_probs, 6)})
sub_df.to_csv(os.path.join(OUT_DIR, "submission_validation.csv"), index=False)

print(f"\n🎉 Saved model to: {OUT_DIR}/model.joblib")
print(f"🎉 Saved validation submission: {OUT_DIR}/submission_validation.csv")

🚀 STARTING THE PIPELINE...
✅ Found Train: /kaggle/input/datasets/amyannjims/odyssey-data/train.csv
✅ Found Validation: /kaggle/input/datasets/amyannjims/odyssey-data/validation.csv

[Amy] Loading data & engineering features...
✅ Features ready in 34.2s! (51 columns)

[Gowri] Training LightGBM model...
✅ Model trained in 23.4s!

[Aein] Finding best threshold to minimize risk penalty...

🏆 VALIDATION RESULTS AT BEST THRESHOLD (T* = 0.0335)
  F1-Score:    0.9638  (Target: > 0.95)
  PR-AUC:      0.9990  (Target: > 0.99)
  Attacks Caught (TP): 44,029 out of 44,040
  Missed Attacks (FN): ONLY 11!
  Total Cost Score:    3,518 (Cost = 20*FN + FP)

🎉 Saved model to: /kaggle/working/model.joblib
🎉 Saved validation submission: /kaggle/working/submission_validation.csv


In [2]:
# ==============================================================================
# AEIN'S CELL: INSTANT SUBMISSION GENERATOR
# ==============================================================================

def make_submission(input_filename, output_filename, expected_rows=None):
    input_path = find_file(input_filename)
    print(f"[*] Processing {input_path} ...")
    
    test_df = load_data(input_path, is_labelled=False)
    print(f"    Loaded {len(test_df):,} test records.")
    
    # 1. Feature Engineering with saved category mappings
    X_test, _ = engineer_features(test_df, cat_mappings=package["cat_mappings"], is_train=False)
    X_test = X_test[package["feature_cols"]]
    
    # 2. Predict Probabilities & Apply Optimal Threshold
    probs = package["model"].predict_proba(X_test)[:, 1]
    th = package["best_threshold"]
    preds = (probs >= th).astype(int)
    
    # 3. Create Valid Submission File
    out_path = os.path.join(OUT_DIR, output_filename)
    sub = pd.DataFrame({
        "row_id": np.arange(1, len(preds) + 1),
        "prediction": preds,
        "probability": np.round(np.clip(probs, 0.0, 1.0), 6)
    })
    sub.to_csv(out_path, index=False)
    
    print(f"✅ Success! Generated {out_path} ({len(sub):,} rows)")
    print(f"    Attacks flagged: {int(np.sum(preds == 1)):,} ({np.mean(preds == 1):.2%})")
    print(f"    Normal passed:   {int(np.sum(preds == 0)):,} ({np.mean(preds == 0):.2%})")
    return out_path

print("✅ Aein's submission pipeline is ready for Hour 4 & Hour 5!")

✅ Aein's submission pipeline is ready for Hour 4 & Hour 5!


In [3]:
# ==========================================
# AMY'S FEATURE AUDIT: WHAT IS DETECTING ATTACKS?
# ==========================================
import pandas as pd

# Check the top 10 features driving our AI
importances = pd.Series(model.feature_importances_, index=list(X_train.columns)).sort_values(ascending=False)
print("🔍 AMY'S TOP 10 INTRUSION DETECTION FEATURES:")
print(importances.head(10))

🔍 AMY'S TOP 10 INTRUSION DETECTION FEATURES:
smeansz              1127
stcpb                1011
sbytes                990
dtcpb                 960
srv_density_src       843
ackdat                820
ct_dst_src_ltm        729
bytes_per_src_pkt     707
ct_srv_src            689
byte_ratio            678
dtype: int32
